In [ ]:
from pathlib import Path
from dotenv import load_dotenv

import chromadb

from llama_index.core import (
    Settings,
    SimpleDirectoryReader,
    StorageContext,
    VectorStoreIndex,
)

from llama_index.core.node_parser import SentenceSplitter

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.llms.openai import OpenAI

from llama_index.vector_stores.chroma import ChromaVectorStore
import os

from llama_index.llms.openai import OpenAI

###########################################################################
# Configuration
###########################################################################

load_dotenv()

DOCUMENT_FOLDER = "../data/knowledge/pdf"

CHROMA_DB = "../data/chroma_db"

COLLECTION_NAME = "future_ai"

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

LLM_MODEL = "deepseek-chat"

CHUNK_SIZE = 512

CHUNK_OVERLAP = 100

TOP_K = 5


###########################################################################
# Configure Models
###########################################################################

Settings.embed_model = HuggingFaceEmbedding(
    model_name=EMBEDDING_MODEL
)



Settings.llm = OpenAI(
    model=LLM_MODEL,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    api_base="https://api.deepseek.com/v1",
    temperature=0,
)

Settings.node_parser = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)


###########################################################################
# Load Documents
###########################################################################

print("Loading documents...")

documents = SimpleDirectoryReader(
    input_dir=DOCUMENT_FOLDER,
    recursive=True
).load_data()

print(f"Loaded {len(documents)} documents")


###########################################################################
# ChromaDB
###########################################################################

client = chromadb.PersistentClient(
    path=CHROMA_DB
)

collection = client.get_or_create_collection(
    COLLECTION_NAME
)

vector_store = ChromaVectorStore(
    chroma_collection=collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)


###########################################################################
# Build Index
###########################################################################

print("Building vector index...")

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
)

print("Finished indexing")


###########################################################################
# Query Engine
###########################################################################

query_engine = index.as_query_engine(
    similarity_top_k=TOP_K
)


###########################################################################
# Chat Loop
###########################################################################

print("\n=========================================")
print("Simple RAG System")
print("Type 'exit' to quit.")
print("=========================================\n")

while True:

    question = input("Question: ")

    if question.lower() == "exit":
        break

    response = query_engine.query(question)

    print("\nAnswer")
    print("----------------------------------------")
    print(response)

    print("\nSources")
    print("----------------------------------------")

    if response.source_nodes:

        for i, node in enumerate(response.source_nodes, start=1):

            print(f"\nSource {i}")
            print(f"Score: {node.score:.3f}")

            print(node.text[:300])
            print("...")

    print("\n")